<a href="https://colab.research.google.com/github/MorreyB/PC-Free/blob/main/Linux_Virtual_PC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [26]:
import os, subprocess, shutil, psutil, time, threading, torch, datetime, socket
import google.colab
from google.colab import drive, output
from IPython.display import HTML, display

# --- 1. INITIALIZE & MOUNT DRIVE ---
drive.mount('/content/drive', force_remount=True)
P_BASE = '/content/drive/MyDrive/colab_desktop_data'
P_WORKSPACE = os.path.join(P_BASE, 'workspace')
os.makedirs(P_WORKSPACE, exist_ok=True)

# Create a symlink in /content for easy access
if not os.path.exists('/content/workspace'):
    os.symlink(P_WORKSPACE, '/content/workspace')

# --- 2. AUTOMATION: BACKGROUND CLEANUP ---
def auto_cleanup_backups(keep=5):
    backup_root = os.path.join(P_BASE, 'session_backups')
    while True:
        if os.path.exists(backup_root):
            backups = sorted([d for d in os.listdir(backup_root) if os.path.isdir(os.path.join(backup_root, d))])
            if len(backups) > keep:
                for i in range(len(backups) - keep):
                    target = os.path.join(backup_root, backups[i])
                    print(f"፤ [Auto-Cleanup] Removing old backup: {backups[i]}")
                    shutil.rmtree(target)
        time.sleep(3600)

threading.Thread(target=auto_cleanup_backups, daemon=True).start()

# --- 3. SYSTEM DEPENDENCIES & FLATPAK ---
print("፤ Installing Dependencies and Flathub...")
subprocess.run("add-apt-repository -y ppa:mozillateam/ppa", shell=True, capture_output=True)
subprocess.run("apt-get update", shell=True, capture_output=True)
subprocess.run("apt-get install -y xfce4 xfce4-goodies tightvncserver novnc python3-websockify sudo firefox flatpak libnvidia-gl-535 libegl1 mesa-utils", shell=True, capture_output=True)
# Add Flathub repository
subprocess.run("flatpak remote-add --if-not-exists flathub https://flathub.org/repo/flathub.flatpakrepo", shell=True)

# --- 4. SESSION MANAGEMENT (ENHANCED) ---
def save_session():
    ts = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    bdir = os.path.join(P_BASE, 'session_backups', ts)
    os.makedirs(bdir, exist_ok=True)
    # Save Configs
    for src, name in [('/root/.vnc', 'vnc_config'), ('/root/.config/xfce4', 'xfce4_settings')]:
        if os.path.exists(src): shutil.copytree(src, os.path.join(bdir, name), dirs_exist_ok=True)
    print(f"፤ Session and Desktop Settings saved to Drive: {ts}")
    print(f"፤ Note: All files in /content/workspace are already safe on Drive.")

def restore_session():
    backup_root = os.path.join(P_BASE, 'session_backups')
    if not os.path.exists(backup_root): return
    backups = sorted([d for d in os.listdir(backup_root)])
    if backups:
        target = os.path.join(backup_root, backups[-1])
        print(f"፤ Restoring Desktop Settings from: {backups[-1]}")
        for dest, name in [('/root/.vnc', 'vnc_config'), ('/root/.config/xfce4', 'xfce4_settings')]:
            src = os.path.join(target, name)
            if os.path.exists(src):
                if os.path.exists(dest): shutil.rmtree(dest)
                shutil.copytree(src, dest)

# --- 5. START VNC ENGINE ---
subprocess.run("pkill -9 -f Xtightvnc|websockify|dbus", shell=True)
restore_session()
!rm -rf /tmp/.X* /tmp/.X11-unix /root/.vnc/*.pid /root/.vnc/*.log
os.environ['USER'] = 'root'
os.makedirs("/root/.vnc", exist_ok=True)
if not os.path.exists("/root/.vnc/passwd"):
    subprocess.run("echo password | vncpasswd -f > /root/.vnc/passwd && chmod 600 /root/.vnc/passwd", shell=True)
with open("/root/.vnc/xstartup", "w") as f:
    f.write("#!/bin/sh\nexport DISPLAY=:10\nexport VGL_DISPLAY=egl\nxset s off\nxset -dpms\ndbus-launch --exit-with-session startxfce4 &\n")
subprocess.run("chmod +x /root/.vnc/xstartup", shell=True)
subprocess.run("vncserver :10 -geometry 1920x1080 -depth 24", shell=True)
subprocess.Popen("websockify --web /usr/share/novnc/ 6080 127.0.0.1:5910", shell=True)

# --- 6. UTILITIES ---
def launch_firefox():
    env = os.environ.copy()
    env.update({"DISPLAY": ":10", "VGL_DISPLAY": "egl"})
    subprocess.Popen(["firefox", "--no-remote"], env=env)

proxy_url = output.eval_js("google.colab.kernel.proxyPort(6080)")
if not proxy_url.endswith('/'): proxy_url += '/'
final_link = f"{proxy_url}vnc.html?autoconnect=true&reconnect=true"
display(HTML(f'<div style="padding:15px; border:2px solid #34a853; border-radius:10px;"><b>Desktop Link:</b> <a href="{final_link}" target="_blank">{final_link}</a><br><b>Persistent Folder:</b> /content/workspace</div>'))
print("፤ All components initialized. Use /content/workspace for permanent files.")

Mounted at /content/drive
፤ Installing Dependencies and Flathub...
፤ Restoring Desktop Settings from: 20260623_135343


፤ All components initialized. Use /content/workspace for permanent files.


### 🚚 Move Stray Files to Permanent Workspace
If you accidentally saved files in the default `/content` folder, use this cell to move them into your persistent Google Drive workspace.

In [17]:
import os, shutil, requests

def setup_persistent_ref():
    ref_url = 'https://raw.githubusercontent.com/vinegarhq/sober/main/flatpak/org.vinegarhq.Sober.flatpakref'
    target_path = os.path.join(P_WORKSPACE, 'org.vinegarhq.Sober.flatpakref')

    print(f"፤ Downloading Sober reference to persistent workspace...")
    r = requests.get(ref_url)
    with open(target_path, 'wb') as f:
        f.write(r.content)

    if os.path.exists(target_path):
        print(f"፤ Success: File is now safe in Drive at: {target_path}")
    else:
        print("፤ Error: Failed to save file to Drive.")

setup_persistent_ref()

፤ Downloading Sober reference to persistent workspace...
፤ Success: File is now safe in Drive at: /content/drive/MyDrive/colab_desktop_data/workspace/org.vinegarhq.Sober.flatpakref


### 🔍 Drive Workspace Verification
This cell confirms that your workspace is correctly linked to your Google Drive for permanent storage.

In [19]:
import os

# Check if the symlink exists and points to Drive
if os.path.islink('/content/workspace'):
    target = os.readlink('/content/workspace')
    print(f"✅ Workspace Link Active: /content/workspace -> {target}")
else:
    print("❌ Workspace link missing. Re-running the Master Cell (9dc8fa53) will fix this.")

# List files currently in the permanent Drive folder
print("\n📂 Files currently safe on Drive:")
!ls -lh /content/workspace

✅ Workspace Link Active: /content/workspace -> /content/drive/MyDrive/colab_desktop_data/workspace

📂 Files currently safe on Drive:
lrwxrwxrwx 1 root root 51 Jun 23 14:00 /content/workspace -> /content/drive/MyDrive/colab_desktop_data/workspace


In [20]:
import shutil
import os

source_file = '/content/org.vinegarhq.Sober.flatpakref'
destination_folder = '/content/workspace'

if os.path.exists(source_file):
    shutil.move(source_file, os.path.join(destination_folder, os.path.basename(source_file)))
    print(f"✅ Successfully moved {source_file} to persistent storage.")
else:
    print("ℹ️ File already moved or not found in temporary storage.")

# Final check of the workspace
print("\n📂 Updated persistent files:")
!ls -lh /content/workspace

ℹ️ File already moved or not found in temporary storage.

📂 Updated persistent files:
lrwxrwxrwx 1 root root 51 Jun 23 14:00 /content/workspace -> /content/drive/MyDrive/colab_desktop_data/workspace


In [21]:
# Launch Sober using the previously installed Flatpak
import subprocess
import os

def launch_sober_persistent():
    env = os.environ.copy()
    env.update({
        "DISPLAY": ":10",
        "VGL_DISPLAY": "egl",
        "__GLX_VENDOR_LIBRARY_NAME": "nvidia"
    })

    print("፤ Launching Sober (Roblox) on the virtual desktop...")
    # Using VirtualGL (vglrun) for GPU acceleration
    subprocess.Popen(["/opt/VirtualGL/bin/vglrun", "-d", "egl", "flatpak", "run", "org.vinegarhq.Sober"], env=env)

launch_sober_persistent()

፤ Launching Sober (Roblox) on the virtual desktop...


In [22]:
import os
import shutil

# 1. Ensure the Desktop directory exists
desktop_path = '/root/Desktop'
os.makedirs(desktop_path, exist_ok=True)

# 2. Copy the flatpakref to the desktop for easy access
source_ref = '/content/workspace/org.vinegarhq.Sober.flatpakref'
dest_ref = os.path.join(desktop_path, 'org.vinegarhq.Sober.flatpakref')

if os.path.exists(source_ref):
    shutil.copy2(source_ref, dest_ref)
    print(f"✅ Copied Sober reference to desktop: {dest_ref}")

# 3. Create a functional Desktop entry to launch Sober directly
shortcut_content = """[Desktop Entry]
Version=1.0
Type=Application
Name=Sober (Roblox)
Comment=Run Sober via VirtualGL
Exec=/opt/VirtualGL/bin/vglrun -d egl flatpak run org.vinegarhq.Sober
Icon=org.vinegarhq.Sober
Terminal=false
Categories=Game;
"""

with open(os.path.join(desktop_path, 'Sober.desktop'), 'w') as f:
    f.write(shortcut_content)

# Set execution permissions
os.chmod(os.path.join(desktop_path, 'Sober.desktop'), 0o755)
print("✅ Created Sober launch shortcut on the desktop.")

✅ Copied Sober reference to desktop: /root/Desktop/org.vinegarhq.Sober.flatpakref
✅ Created Sober launch shortcut on the desktop.


In [23]:
import os
import subprocess

desktop_shortcut = '/root/Desktop/Sober.desktop'

print("--- Sober Shortcut Verification ---")
# 1. Check if the file exists
if os.path.exists(desktop_shortcut):
    print(f"✅ Shortcut File Found: {desktop_shortcut}")

    # 2. Check if the file is executable
    if os.access(desktop_shortcut, os.X_OK):
        print("✅ Shortcut is Executable")
    else:
        print("❌ Shortcut is NOT executable. Fixing now...")
        os.chmod(desktop_shortcut, 0o755)
else:
    print("❌ Shortcut File Missing!")

# 3. Verify Flatpak registration
print("\n--- Application Registration ---")
flatpak_check = subprocess.run(["flatpak", "info", "org.vinegarhq.Sober"], capture_output=True, text=True)
if flatpak_check.returncode == 0:
    print("✅ Sober is correctly installed via Flatpak.")
else:
    print("❌ Sober Flatpak not found. You may need to run the installation cell again.")

# 4. Instructions for the user
print("\n💡 To verify visually: Open your VNC Desktop Link and look for the 'Sober (Roblox)' icon. Double-click it to start working.")

--- Sober Shortcut Verification ---
✅ Shortcut File Found: /root/Desktop/Sober.desktop
✅ Shortcut is Executable

--- Application Registration ---
❌ Sober Flatpak not found. You may need to run the installation cell again.

💡 To verify visually: Open your VNC Desktop Link and look for the 'Sober (Roblox)' icon. Double-click it to start working.


In [25]:
import subprocess

print("፤ Ensuring Flathub remote is active...")
subprocess.run("flatpak remote-add --if-not-exists flathub https://dl.flathub.org/repo/flathub.flatpakrepo", shell=True)

print("፤ Installing Sober (Roblox) from Flathub...")
# Direct installation from the Flathub ID is more reliable than the ref file in some environments
subprocess.run("flatpak install flathub org.vinegarhq.Sober -y", shell=True)

# Final registration check
print("\n--- Registration Check ---")
!flatpak info org.vinegarhq.Sober

፤ Ensuring Flathub remote is active...
፤ Installing Sober (Roblox) from Flathub...

--- Registration Check ---

VinegarHQ & Sober contributors - Play, chat & explore on Roblox

          ID: org.vinegarhq.Sober
         Ref: app/org.vinegarhq.Sober/x86_64/stable
        Arch: x86_64
      Branch: stable
     Version: 1.7.1
     License: LicenseRef-proprietary=https://sober.vinegarhq.org/notice.txt
      Origin: flathub
  Collection: org.flathub.Stable
Installation: system
   Installed: 17.8 MB
     Runtime: org.gnome.Platform/x86_64/50
         Sdk: org.gnome.Sdk/x86_64/50

      Commit: 2dc01ea5b3a80dedebc0b06d7e674b15d8ee3f01fe35115d78d38afd34882327
      Parent: 102964bde9e5c3f14288b75e0ff54777ae06fdea98be073d679b74fa3d9eccb8
     Subject: Update Sober to 2026-06-22_e24358e (1.7.1) (aa724b7e5301)
        Date: 2026-06-22 19:20:03 +0000


In [6]:
# The scraper is now fully configured with the necessary libraries and drivers.
import selenium.webdriver as webdriver
from selenium.webdriver.firefox.service import Service
from selenium.webdriver.firefox.options import Options as FirefoxOptions

def get_scraper():
    firefox_options = FirefoxOptions()
    firefox_options.add_argument('--headless')
    firefox_options.add_argument('--no-sandbox')
    firefox_options.add_argument('--disable-dev-shm-usage')

    service = Service(executable_path='/usr/bin/geckodriver')
    return webdriver.Firefox(service=service, options=firefox_options)

# Usage example:
crawler = get_scraper()
try:
    crawler.get('https://www.google.com/')
    print(f'፤ Scraper active! Page Title: {crawler.title}')
finally:
    crawler.quit()

፤ Scraper active! Page Title: Google


In [70]:
!flatpak install flathub org.vinegarhq.Sober -y

Looking for matches…
Required runtime for org.vinegarhq.Sober/x86_64/stable (runtime/org.gnome.Platform/x86_64/50) found in remote flathub

org.vinegarhq.Sober permissions:
    ipc      network      fallback-x11      pulseaudio     wayland
    dri      devel        tags [1]

    [1] proprietary


        ID                                           Branch      Op Remote  Download
 1.     org.freedesktop.Platform.GL.default          25.08       i  flathub < 142.6 MB
 2.     org.freedesktop.Platform.GL.default          25.08-extra i  flathub < 142.6 MB
 3.     org.freedesktop.Platform.GL.nvidia-580-82-07 1.4         i  flathub < 331.2 MB
 4.     org.freedesktop.Platform.codecs-extra        25.08-extra i  flathub  < 14.4 MB
 5.     org.gnome.Platform.Locale                    50          i  flathub < 386.1 MB
 6.     org.gnome.Platform                           50          i  flathub < 408.5 MB
 7.     org.vinegarhq.Sober                          stable      i  flathub  < 14.9 MB


Instal

In [ ]:
launch_sober()

In [1]:
!apt-get update
!apt install firefox

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,765 kB]
Get:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,307 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [4,028 kB]
Get:13 https://ppa.launchpadcontent.net/deadsnakes

In [39]:
!pip install selenium

In [40]:
import selenium.webdriver as webdriver
from selenium.webdriver.firefox.service import Service
from selenium.webdriver.firefox.options import Options as FirefoxOptions

In [44]:
def selenium_firefox_agent():
    from selenium import webdriver
    from selenium.webdriver.firefox.service import Service
    from selenium.webdriver.firefox.options import Options as FirefoxOptions
    import shutil

    # 1. Ensure geckodriver is installed in the system path
    if not shutil.which('geckodriver'):
        print("፤ Downloading geckodriver...")
        !wget -q https://github.com/mozilla/geckodriver/releases/download/v0.34.0/geckodriver-v0.34.0-linux64.tar.gz
        !tar -xzf geckodriver-v0.34.0-linux64.tar.gz -C /usr/bin
        !rm geckodriver-v0.34.0-linux64.tar.gz

    firefox_options = FirefoxOptions()
    firefox_options.add_argument('--headless')

    # 2. Use Service class for the driver path (Required for Selenium 4.x+)
    service = Service(executable_path='/usr/bin/geckodriver')

    driver = webdriver.Firefox(service=service, options=firefox_options)
    print('፤ Firefox Scraper setup complete!')
    return driver

In [45]:
# Re-run the scraper with the fixed agent
crawler = selenium_firefox_agent()
crawler.get("https://www.google.com/")
print(f"፤ Success! Page Title: {crawler.title}")
source = crawler.page_source
crawler.quit()

፤ Downloading geckodriver...
፤ Firefox Scraper setup complete!
፤ Success! Page Title: Google


In [46]:
# Consolidated Chrome Fix
!apt-get update
!apt-get install -y wget curl unzip libu2f-udev
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt-get install -y ./google-chrome-stable_current_amd64.deb
!pip install -U selenium webdriver-manager

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

chrome_options = Options()
chrome_options.add_argument('--headless')
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')

try:
    # Using Service and ChromeDriverManager is the most reliable way in Colab
    service = Service(ChromeDriverManager().install())
    wd = webdriver.Chrome(service=service, options=chrome_options)
    wd.get("https://www.google.com")
    print(f"፤ Chrome Success! Title: {wd.title}")
    wd.quit()
except Exception as e:
    print(f"፤ Chrome Error: {e}")

Hit:1 https://dl.google.com/linux/chrome-stable/deb stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:5 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:12 https://ppa.launchpadcontent.net/mozillateam/ppa/ubuntu jammy InRelease
Hit:13 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 3,917 B in 1s (2,970 B/s)
Reading package lists... Done
W: Skippin

In [13]:
import os, subprocess, shutil, psutil, time, threading, datetime, socket
from google.colab import drive, output
from IPython.display import HTML, display

# --- 1. INITIALIZE & MOUNT DRIVE ---
drive.mount('/content/drive', force_remount=True)
P_BASE = '/content/drive/MyDrive/colab_desktop_data'
os.makedirs(P_BASE, exist_ok=True)

# --- 2. AUTOMATION: BACKGROUND CLEANUP ---
def auto_cleanup_backups(keep=5):
    backup_root = os.path.join(P_BASE, 'session_backups')
    while True:
        if os.path.exists(backup_root):
            backups = sorted([d for d in os.listdir(backup_root) if os.path.isdir(os.path.join(backup_root, d))])
            if len(backups) > keep:
                for i in range(len(backups) - keep):
                    shutil.rmtree(os.path.join(backup_root, backups[i]))
        time.sleep(3600)

threading.Thread(target=auto_cleanup_backups, daemon=True).start()

# --- 3. SYSTEM DEPENDENCIES & GPU ---
print("፤ Installing Dependencies (Firefox, XFCE, VNC, VirtualGL)...")
subprocess.run("add-apt-repository -y ppa:mozillateam/ppa", shell=True, capture_output=True)
with open('/etc/apt/preferences.d/99mozillateam', 'w') as f:
    f.write('Package: firefox*\nPin: release o=LP-PPA-mozillateam\nPin-Priority: 1001\n')
subprocess.run("apt-get update", shell=True, capture_output=True)
subprocess.run("apt-get install -y xfce4 xfce4-goodies tightvncserver novnc python3-websockify sudo firefox flatpak libnvidia-gl-535 libegl1 mesa-utils", shell=True, capture_output=True)

if not shutil.which("vglrun"):
    subprocess.run("wget https://sourceforge.net/projects/virtualgl/files/3.1/virtualgl_3.1_amd64.deb/download -O vgl.deb", shell=True, capture_output=True)
    subprocess.run("dpkg -i vgl.deb && apt-get install -f -y", shell=True, capture_output=True)

subprocess.run("ln -sf /usr/lib/x86_64-linux-gnu/libEGL_nvidia.so.0 /usr/lib/x86_64-linux-gnu/libEGL.so.1", shell=True)
subprocess.run("ldconfig", shell=True)

# --- 4. SESSION MANAGEMENT ---
def save_session():
    ts = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    bdir = os.path.join(P_BASE, 'session_backups', ts)
    os.makedirs(bdir, exist_ok=True)
    for src, name in [('/root/.vnc', 'vnc_config'), ('/root/.config/xfce4', 'xfce4_settings')]:
        if os.path.exists(src): shutil.copytree(src, os.path.join(bdir, name), dirs_exist_ok=True)
    print(f"፤ Session saved: {ts}")

def restore_session():
    backup_root = os.path.join(P_BASE, 'session_backups')
    if os.path.exists(backup_root):
        backups = sorted([d for d in os.listdir(backup_root)])
        if backups:
            target = os.path.join(backup_root, backups[-1])
            for dest, name in [('/root/.vnc', 'vnc_config'), ('/root/.config/xfce4', 'xfce4_settings')]:
                src = os.path.join(target, name)
                if os.path.exists(src):
                    if os.path.exists(dest): shutil.rmtree(dest)
                    shutil.copytree(src, dest)

# --- 5. CLEAN & START VNC SERVER ---
print("፤ Starting VNC & Proxy...")
subprocess.run("pkill -9 -f Xtightvnc|websockify|dbus", shell=True)
!rm -rf /tmp/.X10-lock /tmp/.X11-unix/X10
restore_session()

os.environ['USER'] = 'root'
os.makedirs("/root/.vnc", exist_ok=True)
if not os.path.exists("/root/.vnc/passwd"):
    subprocess.run("echo password | vncpasswd -f > /root/.vnc/passwd && chmod 600 /root/.vnc/passwd", shell=True)
with open("/root/.vnc/xstartup", "w") as f:
    f.write("#!/bin/sh\nexport DISPLAY=:10\nexport VGL_DISPLAY=egl\ndbus-launch --exit-with-session startxfce4 &\n")
subprocess.run("chmod +x /root/.vnc/xstartup", shell=True)
subprocess.run("vncserver :10 -geometry 1920x1080 -depth 24", shell=True)
subprocess.Popen("websockify --web /usr/share/novnc/ 6080 127.0.0.1:5910", shell=True)
time.sleep(2)

# --- 6. APP LAUNCHERS ---
def launch_firefox():
    env = os.environ.copy()
    env.update({"DISPLAY": ":10", "VGL_DISPLAY": "egl", "__GLX_VENDOR_LIBRARY_NAME": "nvidia", "LD_PRELOAD": "/usr/lib/x86_64-linux-gnu/libGL.so.1"})
    subprocess.Popen(["/opt/VirtualGL/bin/vglrun", "-d", "egl", "firefox", "--no-remote"], env=env)

def launch_sober():
    env = os.environ.copy()
    env.update({"DISPLAY": ":10", "VGL_DISPLAY": "egl", "__GLX_VENDOR_LIBRARY_NAME": "nvidia"})
    subprocess.Popen(["/opt/VirtualGL/bin/vglrun", "-d", "egl", "flatpak", "run", "org.sober.Sober"], env=env)

# --- 7. REFRESH MENU & DISPLAY LINK ---
subprocess.run("update-desktop-database /usr/share/applications", shell=True)
os.makedirs("/root/Desktop", exist_ok=True)
!ln -sf /usr/share/applications/firefox.desktop /root/Desktop/Firefox.desktop

proxy_url = google.colab.output.eval_js("google.colab.kernel.proxyPort(6080)")
if not proxy_url.endswith('/'): proxy_url += '/'
final_link = f"{proxy_url}vnc.html?autoconnect=true&reconnect=true"

display(HTML(f'''<div style="padding:20px; border:3px solid #34a853; border-radius:10px; background-color: #f6fdf9;">\n    <h3 style="margin-top:0; color: #34a853;">Environment Ready</h3>\n    <p>Click below to open your session:</p>\n    <a href="{final_link}" target="_blank" style="font-weight:bold; font-size:1.1em;">{final_link}</a>\n</div>'''))

launch_firefox()
print("፤ All components initialized.")

Mounted at /content/drive
፤ Installing Dependencies (Firefox, XFCE, VNC, VirtualGL)...
፤ Starting VNC & Proxy...


፤ All components initialized.


### 💾 Manual Workspace Backup
Run this cell to immediately sync your current XFCE settings and VNC configuration to your Google Drive. This ensures that even if the session ends abruptly, your desktop layout will be preserved.

In [8]:
save_session()

# Verification: List the contents of your backup folder
import os
backup_root = os.path.join(P_BASE, 'session_backups')
if os.path.exists(backup_root):
    latest_backups = sorted(os.listdir(backup_root))
    print(f"\n፤ Current backups on Drive: {latest_backups}")
else:
    print("፤ Error: Backup directory not found on Drive.")

፤ Session saved: 20260623_135343

፤ Current backups on Drive: ['20260619_114346', '20260623_124623', '20260623_135343']


### ⏪ Restore Workspace from Specific Timestamp
If you want to go back to a specific version of your desktop settings, list your backups and enter the timestamp below.

In [11]:
def restore_from_timestamp(timestamp):
    backup_root = os.path.join(P_BASE, 'session_backups')
    target = os.path.join(backup_root, timestamp)

    if not os.path.exists(target):
        print(f"፤ Error: Backup '{timestamp}' not found.")
        return

    print(f"፤ Restoring desktop from: {timestamp}...")
    for dest, name in [('/root/.vnc', 'vnc_config'), ('/root/.config/xfce4', 'xfce4_settings')]:
        src = os.path.join(target, name)
        if os.path.exists(src):
            if os.path.exists(dest): shutil.rmtree(dest)
            shutil.copytree(src, dest)

    print("፤ Restore complete. Please run the Master Cell (9dc8fa53) to restart the VNC session with these settings.")

# --- USER INPUT ---
# List available timestamps
backups = sorted(os.listdir(os.path.join(P_BASE, 'session_backups')))
print(f"Available backups: {backups}")

# Paste the timestamp you want here:
CHOSEN_BACKUP = '20260623_135343'

# Run the restoration
restore_from_timestamp(CHOSEN_BACKUP)

Available backups: ['20260619_114346', '20260623_124623', '20260623_135343']
፤ Restoring desktop from: 20260623_135343...
፤ Restore complete. Please run the Master Cell (9dc8fa53) to restart the VNC session with these settings.


### 📂 Persistence Tips
1. **Always use `/content/drive/MyDrive/...`**: Any files created in the default `/content` folder will be deleted when you disconnect.
2. **Configuration Persistence**: The Master Cell is already configured to look for the last saved session in your Drive and restore it automatically during setup.

In [30]:
!bash

bash: cannot set terminal process group (3104): Inappropriate ioctl for device
bash: no job control in this shell
/content# !apt-get update !apt install firefox
bash: !apt: event not found
/content# !apt-get update
bash: !apt: event not found



/content# ^C


In [ ]:
import socket
import psutil

def check_port(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(('127.0.0.1', port)) == 0

print(f"፤ Checking VNC Server (5910): {'RUNNING' if check_port(5910) else 'NOT FOUND'}")
print(f"፤ Checking noVNC Proxy (6080): {'RUNNING' if check_port(6080) else 'NOT FOUND'}")

print("\n፤ Active Processes:")
for proc in psutil.process_iter(['pid', 'name', 'cmdline']):
    if any(x in (proc.info['name'] or '') for x in ['Xtightvnc', 'websockify']):
        print(f" - PID {proc.info['pid']}: {proc.info['name']} ({' '.join(proc.info['cmdline'] or [])})")

፤ Checking VNC Server (5910): NOT FOUND
፤ Checking noVNC Proxy (6080): RUNNING

፤ Active Processes:
 - PID 20946: websockify (/usr/bin/python3 /usr/bin/websockify --web /usr/share/novnc/ 6080 127.0.0.1:5910)


In [ ]:
import subprocess, os, time

print("፤ Cleaning up stale VNC locks...")
# Remove existing lock files that might prevent VNC from starting
!rm -rf /tmp/.X10-lock /tmp/.X11-unix/X10
subprocess.run("pkill -9 -f Xtightvnc", shell=True)

print("፤ Starting VNC Server on :10...")
os.environ['USER'] = 'root'
# Start the VNC server
res = subprocess.run("vncserver :10 -geometry 1920x1080 -depth 24", shell=True, capture_output=True, text=True)
print(res.stdout)

# Verify if it's actually listening now
import socket
with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
    is_up = s.connect_ex(('127.0.0.1', 5910)) == 0
    print(f"\n፤ VNC Server Status: {'SUCCESSFULLY STARTED' if is_up else 'FAILED TO START'}")

if not is_up:
    print("፤ Error Log:", res.stderr)

፤ Cleaning up stale VNC locks...
፤ Starting VNC Server on :10...


፤ VNC Server Status: SUCCESSFULLY STARTED


In [ ]:
import shutil
import subprocess

def ensure_firefox_installed():
    if shutil.which('firefox'):
        print("፤ Firefox is already installed.")
    else:
        print("፤ Firefox not found. Starting installation...")
        # Add PPA for the latest Firefox ESR/Stable
        subprocess.run("add-apt-repository -y ppa:mozillateam/ppa", shell=True)
        subprocess.run("apt-get update", shell=True)
        # Ensure we prioritize the PPA over the snap-stub
        with open('/etc/apt/preferences.d/99mozillateam', 'w') as f:
            f.write('Package: firefox*\nPin: release o=LP-PPA-mozillateam\nPin-Priority: 1001\n')
        subprocess.run("apt-get install -y firefox", shell=True)
        print("፤ Firefox installation complete.")

ensure_firefox_installed()

፤ Firefox is already installed.


In [ ]:
def startup_firefox():
    """
    Ensures Firefox is installed and launches it with VirtualGL on the current desktop session.
    """
    import shutil
    import subprocess
    import os

    # 1. Ensure Installation
    if not shutil.which('firefox'):
        print("፤ Firefox not found. Installing via PPA...")
        subprocess.run("add-apt-repository -y ppa:mozillateam/ppa", shell=True, capture_output=True)
        subprocess.run("apt-get update", shell=True, capture_output=True)
        with open('/etc/apt/preferences.d/99mozillateam', 'w') as f:
            f.write('Package: firefox*\nPin: release o=LP-PPA-mozillateam\nPin-Priority: 1001\n')
        subprocess.run("apt-get install -y firefox", shell=True, capture_output=True)
        print("፤ Firefox installation complete.")
    else:
        print("፤ Firefox is already installed.")

    # 2. Launch with GPU Acceleration
    print("፤ Launching Firefox...")
    env = os.environ.copy()
    env.update({
        "DISPLAY": ":10",
        "VGL_DISPLAY": "egl",
        "__GLX_VENDOR_LIBRARY_NAME": "nvidia",
        "LD_PRELOAD": "/usr/lib/x86_64-linux-gnu/libGL.so.1"
    })

    # Use vglrun if available for hardware acceleration
    if os.path.exists("/opt/VirtualGL/bin/vglrun"):
        subprocess.Popen(["/opt/VirtualGL/bin/vglrun", "-d", "egl", "firefox", "--no-remote"], env=env)
    else:
        subprocess.Popen(["firefox", "--no-remote"], env=env)

# Run the startup function
startup_firefox()

፤ Firefox is already installed.
፤ Launching Firefox...


In [ ]:
import subprocess
import os

print("፤ Refreshing application menu database...")
# Update the desktop database for the current user
subprocess.run("update-desktop-database /usr/share/applications", shell=True)

# Check if the firefox.desktop file exists
shortcut_path = "/usr/share/applications/firefox.desktop"
if os.path.exists(shortcut_path):
    print(f"፤ Found Firefox shortcut at: {shortcut_path}")
    # Force a refresh of the XFCE panel
    subprocess.run("xfce4-panel --restart", shell=True, env=os.environ.copy())
    print("፤ XFCE Panel restarted. Please check the Application Launcher again.")
else:
    print("፤ Warning: Firefox desktop file not found in /usr/share/applications.")

# Fallback: Create a symlink to the desktop if it's missing from the menu
!ln -sf /usr/share/applications/firefox.desktop /root/Desktop/Firefox.desktop
print("፤ Added a fallback Firefox icon directly to your Desktop.")

፤ Refreshing application menu database...
፤ Found Firefox shortcut at: /usr/share/applications/firefox.desktop
፤ XFCE Panel restarted. Please check the Application Launcher again.
፤ Added a fallback Firefox icon directly to your Desktop.


In [ ]:
import subprocess, os, time, socket, google.colab.output
from IPython.display import HTML, display

print("፤ Cleaning stale VNC processes and locks...")
# Kill existing VNC and Websockify
subprocess.run("pkill -9 -f Xtightvnc|websockify", shell=True)
# Remove lock files
!rm -rf /tmp/.X10-lock /tmp/.X11-unix/X10

print("፤ Restarting VNC Server...")
os.environ['USER'] = 'root'
subprocess.run("vncserver :10 -geometry 1920x1080 -depth 24", shell=True, capture_output=True)

print("፤ Restarting noVNC Proxy...")
subprocess.Popen("websockify --web /usr/share/novnc/ 6080 127.0.0.1:5910", shell=True)
time.sleep(2)

# Verify connection
with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
    is_up = s.connect_ex(('127.0.0.1', 5910)) == 0

if is_up:
    proxy_url = google.colab.output.eval_js("google.colab.kernel.proxyPort(6080)")
    if not proxy_url.endswith('/'): proxy_url += '/'
    new_link = f"{proxy_url}vnc.html?autoconnect=true&reconnect=true"

    display(HTML(f'''<div style="padding:20px; border:3px solid #34a853; border-radius:10px; background-color: #f6fdf9;">
        <h3 style="margin-top:0; color: #34a853;">Success: Server Restored</h3>
        <p>Click the link below to reconnect:</p>
        <a href="{new_link}" target="_blank" style="font-weight:bold; font-size:1.1em;">{new_link}</a>
    </div>'''))
else:
    print("፤ Critical: VNC Server failed to start. Please check if the 'Master Cell' ran correctly.")

፤ Cleaning stale VNC processes and locks...
፤ Restarting VNC Server...
፤ Restarting noVNC Proxy...


In [ ]:
launch_sober()

፤ Sober starting...


In [ ]:
# This cell is now redundant and can be removed.

In [ ]:
# This cell is now redundant and can be removed.

In [ ]:
# To start apps, use the functions defined in the master cell above:
# launch_firefox()
# launch_sober()
# save_session()

In [ ]:
save_session()

፤ Session saved: 20260619_114346


In [ ]:
import os
backup_path = '/content/drive/MyDrive/colab_desktop_data/session_backups/'
if os.path.exists(backup_path):
    display(os.listdir(backup_path))
else:
    print('Backup directory not found.')

['20260619_114346']

In [ ]:
import google.colab.output

# --- RESTART PROXY ---
# Kill existing proxy if any
!pkill -9 -f websockify

# Start websockify in the background
import subprocess
subprocess.Popen("websockify --web /usr/share/novnc/ 6080 127.0.0.1:5910", shell=True)

# Wait a moment for it to initialize
import time
time.sleep(2)

# Generate the fresh Proxy URL
proxy_url = google.colab.output.eval_js("google.colab.kernel.proxyPort(6080)")
# Ensure there is a trailing slash before vnc.html
if not proxy_url.endswith('/'):
    proxy_url += '/'

final_vnc_link = f"{proxy_url}vnc.html?autoconnect=true&reconnect=true"

from IPython.display import HTML, display
display(HTML(f'''<div style="padding:20px; border:3px solid #4285f4; border-radius:10px; background-color: #f8f9fa;">
    <h3 style="margin-top:0; color: #4285f4;">New Desktop Connection</h3>
    <p>The link has been fixed. Click below to open the desktop:</p>
    <a href="{final_vnc_link}" target="_blank" style="font-weight:bold; font-size:1.1em;">{final_vnc_link}</a>
</div>'''))

In [ ]:
launch_firefox()

፤ Firefox starting...


In [ ]:
launch_firefox()

፤ Firefox starting...


In [ ]:
import shutil
import os

backup_root = os.path.join(P_BASE, 'session_backups')
if os.path.exists(backup_root):
    # Get all subdirectories in the backup folder, excluding any non-timestamped files
    backups = sorted([d for d in os.listdir(backup_root) if os.path.isdir(os.path.join(backup_root, d))])

    if len(backups) > 1:
        oldest_backup = backups[0]
        target_path = os.path.join(backup_root, oldest_backup)
        print(f"፤ Deleting oldest backup: {oldest_backup}")
        shutil.rmtree(target_path)
        print("፤ Deletion complete.")
    else:
        print("፤ Only one or zero backups found. Skipping deletion to prevent data loss.")
else:
    print("፤ Backup directory not found.")

፤ Deleting oldest backup: 20260619_110854
፤ Deletion complete.
